In [1]:
import os
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [5]:
import os
#local OS인 경우와 colap 인 경우 디렉토리 설정은 각각 다음과 같음
imdb_dir = 'D:\\AIWork\\Data\\aclImdb\\aclImdb'
#imdb_dir = '/content/drive/MyDrive/Colab Notebooks/dataset/aclImdb’
train_dir = os.path.join(imdb_dir, 'train')
labels = []
texts = []


In [6]:
# 데이터 로드 및 라벨링
for label_type in ['neg', 'pos']:
    dir_name = os.path.join(train_dir, label_type)
    print(dir_name)  # 확인용 출력
    for fname in os.listdir(dir_name):
        if fname[-4:] == '.txt':
            with open(os.path.join(dir_name, fname), encoding='utf8') as f:
                texts.append(f.read())
            labels.append(0 if label_type == 'neg' else 1)

print(texts[0])      # 부정의 첫 번째 문장
print(labels[0])     # 부정의 첫 번째 label
print(texts[12500])  # 긍정의 첫 번째 문장
print(labels[12500]) # 긍정의 첫 번째 label

D:\AIWork\Data\aclImdb\aclImdb\train\neg
D:\AIWork\Data\aclImdb\aclImdb\train\pos
Story of a man who has unnatural feelings for a pig. Starts out with a opening scene that is a terrific example of absurd comedy. A formal orchestra audience is turned into an insane, violent mob by the crazy chantings of it's singers. Unfortunately it stays absurd the WHOLE time with no general narrative eventually making it just too off putting. Even those from the era should be turned off. The cryptic dialogue would make Shakespeare seem easy to a third grader. On a technical level it's better than you might think with some good cinematography by future great Vilmos Zsigmond. Future stars Sally Kirkland and Frederic Forrest can be seen briefly.
0
Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scr

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 데이터 전처리 설정
maxlen = 200 # 200개 단어 이후는 버립니다
training_samples = 10000
validation_samples = 10000
max_words = 10000 # 데이터셋에서 가장 빈도 높은 10,000개의 단어만 사용합니다



# 토큰화 및 패딩
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
word_index = tokenizer.word_index
print('%s개의 고유한 토큰을 찾았습니다.' % len(word_index))

data = pad_sequences(sequences, maxlen=maxlen)
labels = np.asarray(labels)
print('데이터 텐서의 크기:', data.shape)
print('레이블 텐서의 크기:', labels.shape)

88582개의 고유한 토큰을 찾았습니다.
데이터 텐서의 크기: (25000, 200)
레이블 텐서의 크기: (25000,)


In [8]:
# 데이터 셔플 및 분할
indices = np.arange(data.shape[0])
np.random.shuffle(indices)
data = data[indices]
labels = labels[indices]

x_train = data[:training_samples]
y_train = labels[:training_samples]
x_val = data[training_samples: training_samples + validation_samples]
y_val = labels[training_samples: training_samples + validation_samples]

# 모델 정의
model = Sequential()
model.add(Embedding(max_words, 64, input_length=maxlen))
model.add(Bidirectional(LSTM(64, return_sequences=True)))
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(32)))
model.add(Dense(1, activation='sigmoid'))

C:\Users\fasoft\anaconda3\envs\tf_env\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [9]:
# 모델 컴파일
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])

# 콜백 정의
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_loss')

# 모델 학습
history = model.fit(x_train, y_train, epochs=20, batch_size=64, validation_data=(x_val, y_val),
                    callbacks=[early_stopping, model_checkpoint])

Epoch 1/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 122s 700ms/step - acc: 0.7169 - loss: 0.5328 - val_acc: 0.8258 - val_loss: 0.4233
Epoch 2/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 73s 463ms/step - acc: 0.8881 - loss: 0.2825 - val_acc: 0.8296 - val_loss: 0.4413
Epoch 3/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 77s 493ms/step - acc: 0.9442 - loss: 0.1618 - val_acc: 0.8556 - val_loss: 0.4090
Epoch 4/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 81s 514ms/step - acc: 0.9668 - loss: 0.1058 - val_acc: 0.8477 - val_loss: 0.5002
Epoch 5/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 74s 475ms/step - acc: 0.9738 - loss: 0.0790 - val_acc: 0.8359 - val_loss: 0.5640
Epoch 6/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 76s 483ms/step - acc: 0.9841 - loss: 0.0540 - val_acc: 0.8426 - val_loss: 0.5742
Epoch 7/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 77s 492ms/step - acc: 0.9908 - loss: 0.0333 - val_acc: 0.8302 - val_loss: 0.6580
Epoch 8/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 72s 462ms/step - acc: 0.9892 - loss: 0.0340 - val_acc: 0.8392 - val_loss: 0.7033


In [10]:
# 모델을 사용하여 예측
test_texts = [
    "I hated this movie. It was terrible and the acting was horrible.",
    "This was the worst film I have ever seen. Not worth the time.",
    "I loved this movie. It was fantastic and the acting was great.",
    "This was the best film I have seen in a long time. Totally worth it."
]
test_sequences = tokenizer.texts_to_sequences(test_texts)
test_data = pad_sequences(test_sequences, maxlen=maxlen)

# 저장된 모델 로드
from tensorflow.keras.models import load_model
best_model = load_model('best_model.keras', custom_objects=None, compile=True)

predictions = best_model.predict(test_data)

for i, test_text in enumerate(test_texts):
    print(f"Text: {test_text}")
    print(f"Prediction: {'Positive' if predictions[i] > 0.5 else 'Negative'} (확률: {predictions[i][0]:.4f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Text: I hated this movie. It was terrible and the acting was horrible.
Prediction: Negative (확률: 0.0133)
Text: This was the worst film I have ever seen. Not worth the time.
Prediction: Negative (확률: 0.0326)
Text: I loved this movie. It was fantastic and the acting was great.
Prediction: Negative (확률: 0.2693)
Text: This was the best film I have seen in a long time. Totally worth it.
Prediction: Negative (확률: 0.4266)


In [11]:
# 모델을 사용하여 예측
test_texts = [
    "I hated this movie. It was terrible and the acting was horrible.",
    "This was the worst film I have ever seen. Not worth the time.",
    "I loved this movie. It was fantastic and the acting was great.",
    "This was the best film I have seen in a long time. Totally worth it.",
    "I had high hopes for this movie, but it was a complete letdown. The storyline was incredibly dull, and the pacing was excruciatingly slow. The characters lacked depth, making it hard to care about their fates. Overall, a waste of time.",
    "This film was a disaster from start to finish. The dialogue was cringe-worthy, and the special effects looked outdated. I couldn't believe how poorly executed the entire production was. Definitely one to avoid.",
    "What an amazing film! The plot was deeply engaging, keeping me on the edge of my seat throughout. The actors delivered powerful performances that brought the story to life. This movie is a must-watch for everyone.",
    "I was thoroughly impressed by this film. The cinematography was stunning, capturing the beauty of every scene. The story was heartwarming and inspiring, and the music perfectly complemented the mood. An outstanding cinematic experience."
]
test_sequences = tokenizer.texts_to_sequences(test_texts)
test_data = pad_sequences(test_sequences, maxlen=maxlen)

# 저장된 모델 로드
from tensorflow.keras.models import load_model
best_model = load_model('best_model.keras', custom_objects=None, compile=True)

predictions = best_model.predict(test_data)

for i, test_text in enumerate(test_texts):
    #print(f"Text: {test_text}")
    print(f"Text: {test_text[:20]}...")  # 처음 20글자만 출력
    print(f"Prediction: {'Positive' if predictions[i] > 0.5 else 'Negative'} (확률: {predictions[i][0]:.4f})")
    #print(f"Prediction: {'Positive' if predictions[i] > 0.5 else 'Negative'} (확률: {predictions[i][0]:.4f})")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Text: I hated this movie. ...
Prediction: Negative (확률: 0.0133)
Text: This was the worst f...
Prediction: Negative (확률: 0.0326)
Text: I loved this movie. ...
Prediction: Negative (확률: 0.2693)
Text: This was the best fi...
Prediction: Negative (확률: 0.4266)
Text: I had high hopes for...
Prediction: Negative (확률: 0.0080)
Text: This film was a disa...
Prediction: Negative (확률: 0.0113)
Text: What an amazing film...
Prediction: Positive (확률: 0.9883)
Text: I was thoroughly imp...
Prediction: Positive (확률: 0.9881)
